# Dinámica 4: Comparación bilingüe

## ¿Son los modelos de NER lingüísticamente neutrales?

Un modelo de NER no es una herramienta universal: es el producto de una decisión de diseño sobre qué constituye una entidad, qué lengua se usa y qué corpus de entrenamiento se emplea. Dos modelos entrenados en lenguas distintas no solo tienen vocabularios distintos: tienen **ontologías distintas**, esto es, clasifican el mundo de manera diferente.

Esta dinámica compara el comportamiento de un modelo en español y uno en inglés sobre el mismo contenido.

## Objetivo

1. Observar si las mismas entidades son detectadas en las dos lenguas.
2. Comparar las etiquetas: ¿las categorías son equivalentes entre modelos?
3. Reflexionar sobre qué implica esto para un profesional que trabaja con textos multilingües.

In [ ]:
!python -m spacy download es_core_news_sm --quiet
!python -m spacy download en_core_web_sm --quiet
import spacy
import pandas as pd
from spacy import displacy

nlp_es = spacy.load("es_core_news_sm")
nlp_en = spacy.load("en_core_web_sm")
print("Modelos cargados: español e inglés.")

## Paso 1: El texto en español

Pega a continuación un fragmento de texto en español. Puede ser la noticia que usaste en la dinámica anterior u otro texto con varios tipos de entidades.

In [ ]:
texto_es = """
PEGA AQUÍ TU TEXTO EN ESPAÑOL
"""

if texto_es.strip() == "PEGA AQUÍ TU TEXTO EN ESPAÑOL":
    print("Por favor, pega el texto en español.")
else:
    doc_es = nlp_es(texto_es)
    print("Entidades detectadas en español:")
    displacy.render(doc_es, style="ent", jupyter=True)

## Paso 2: El mismo contenido en inglés

Pega a continuación la traducción al inglés del mismo texto. Puede ser una traducción propia o generada con DeepL o Google Translate.

In [ ]:
texto_en = """
PASTE HERE YOUR TEXT IN ENGLISH
"""

if texto_en.strip() == "PASTE HERE YOUR TEXT IN ENGLISH":
    print("Por favor, pega el texto en inglés.")
else:
    doc_en = nlp_en(texto_en)
    print("Entidades detectadas en inglés:")
    displacy.render(doc_en, style="ent", jupyter=True)

## Paso 3: Las etiquetas no son equivalentes

Antes de comparar los resultados, conviene observar que los dos modelos no usan las mismas categorías:

In [ ]:
tabla_etiquetas = [
    ("PER",  "Persona",                   "PERSON", "Persona"),
    ("ORG",  "Organización",              "ORG",    "Organización"),
    ("LOC",  "Lugar (físico y político)", "GPE",    "Entidad geopolítica (país, ciudad, estado)"),
    ("LOC",  "Lugar (físico y político)", "LOC",    "Lugar físico (montaña, río, región)"),
    ("MISC", "Miscelánea",                "NORP",   "Nacionalidades, grupos religiosos o políticos"),
    ("-",    "-",                         "DATE",   "Fechas y periodos"),
    ("-",    "-",                         "MONEY",  "Valores monetarios"),
]

df_etiquetas = pd.DataFrame(
    tabla_etiquetas,
    columns=["Etiqueta ES", "Significado ES", "Etiqueta EN", "Significado EN"]
)
print(df_etiquetas.to_string(index=False))

Nótese que el modelo inglés distingue entre `GPE` (entidad geopolítica: país, ciudad, estado) y `LOC` (localización física: montaña, río, región), mientras que el español agrupa ambos bajo `LOC`. Esta diferencia no es arbitraria: refleja decisiones de diseño sobre qué distinciones conceptuales vale la pena capturar, tomadas por equipos distintos con corpus distintos.

## Paso 4: Comparación de entidades detectadas

In [ ]:
entidades_es = sorted(set((ent.text.strip(), ent.label_) for ent in doc_es.ents))
entidades_en = sorted(set((ent.text.strip(), ent.label_) for ent in doc_en.ents))

df_es = pd.DataFrame(entidades_es, columns=["Texto", "Etiqueta"])
df_en = pd.DataFrame(entidades_en, columns=["Texto", "Etiqueta"])

print("=== Entidades detectadas en español ===")
print(df_es.to_string(index=False))
print("\n=== Entidades detectadas en inglés ===")
print(df_en.to_string(index=False))

## Análisis

1. ¿Hay entidades detectadas en un idioma que no se detectan en el otro? ¿A qué puede deberse?
2. ¿Cómo afecta la diferencia entre `LOC` y `GPE` a la interpretación de los resultados en un flujo de trabajo de traducción?
3. Si tuvieras que decidir qué modelo usar para analizar un corpus bilingüe español-inglés, ¿qué criterios aplicarías?

---

**Para reflexionar:** Un sistema de NER no es una descripción objetiva del texto: es la proyección de una serie de decisiones de diseño (qué cuenta como entidad, qué corpus se usó, qué lengua se privilegia) sobre el texto. Esa proyección puede ser más o menos adecuada según el dominio y la tarea.